# UFC Fight Data Web Scraping and Analysis

## Table of Contents
1. [Project Overview](#project-overview)
2. [Dependencies](#dependencies)
3. [Data Collection](#data-collection)
   - [Initialize Variables](#initialize-variables)
   - [Define Scrapy Spider](#define-the-scrapy-spider)
   - [Run Scrapy Spider](#run-the-scrapy-spider)
4. [Data Processing](#data-processing)
   - [Load and Inspect Data](#load-and-inspect-data)
   - [Data Cleaning](#data-cleaning)
   - [Feature Engineering](#feature-engineering)
5. [Data Overview](#data-overview)

## Project Overview

This notebook scrapes UFC fight statistics and event data from [ufcstats.com](http://ufcstats.com) using Python and Scrapy.

### Features
- Scrapes event pages from UFCStats.com
- Extracts fight details, fighter information, and statistics
- Processes and cleans the data
- Creates new features for analysis
- Saves processed data to CSV files

## Dependencies

Required Python packages:
- pandas: Data manipulation and storage
- scrapy: Web scraping framework
- numpy: Numerical operations
- jupyter: Interactive notebook interface

In [1]:
# Import required libraries
import pandas as pd
import scrapy
from scrapy.crawler import CrawlerProcess

## Data Collection

### Initialize Variables

Set up the parameters and data structures for the scraping process:

In [2]:
# Number of pages to scrape from UFCStats.com
pages = 1

# Lists to store scraped data
event_links = []  # Store event URLs
fights = []       # Store fight details

### Define the Scrapy Spider

The `UfcSpider` class handles the web scraping process with three main methods:
1. `start_requests`: Initiates scraping from the main events page
2. `parse_events`: Extracts event details and fight links
3. `parse_fights`: Collects detailed fight statistics

In [3]:
class UfcSpider(scrapy.Spider):
    name = "ufc_spider"

    def start_requests(self):
        """Start scraping from the main events page"""
        for p in range(1, pages + 1):
            url = f"http://ufcstats.com/statistics/events/completed?page={p}"
            yield scrapy.Request(url=url, callback=self.parse_main)

    def parse_main(self, response):
        """Extract event links from the main page"""
        event_links_on_page = response.css("a.b-link.b-link_style_black::attr(href)").extract()
        for e in event_links_on_page:
            event_links.append({"event_link": e})
            yield response.follow(url=e, callback=self.parse_events)

    def parse_events(self, response):
        """Extract event details and fight links"""
        fight_links = response.css("a.b-flag.b-flag_style_green::attr(href)").extract()
        date = response.css("li.b-list__box-list-item:nth-child(1)::text").extract()[1]
        location = response.css("li.b-list__box-list-item:nth-child(2)::text").extract()[1]

        for f in fight_links:
            yield response.follow(
                url=f,
                callback=self.parse_fights,
                meta={"date": date, "location": location},
            )

    def parse_fights(self, response):
        """Extract detailed fight statistics"""
        date = response.meta["date"]
        location = response.meta["location"]

        # Extract fighter information
        fighter_details = response.css("div.b-fight-details__person")
        win_loss_1, win_loss_2 = fighter_details.css("i.b-fight-details__person-status::text").extract()[0:2]
        name_1, name_2 = fighter_details.css("h3 > a::text").extract()[0:2]
        stage_name_1, stage_name_2 = fighter_details.css("p.b-fight-details__person-title::text").extract()[0:2]

        # Extract fight details
        fight_details = response.css("div.b-fight-details__content")
        method = fight_details.css("p:nth-child(1) > i.b-fight-details__text-item_first > i:nth-child(2)::text").get()
        round_num = fight_details.css("p:nth-child(1) > i:nth-child(2)::text").extract()[1]
        time = fight_details.css("p:nth-child(1) > i:nth-child(3)::text").extract()[1]
        time_format = fight_details.css("p:nth-child(1) > i:nth-child(4)::text").extract()[1]
        referee = fight_details.css("p:nth-child(1) > i:nth-child(5) > span::text").get()
        details = fight_details.css("p:nth-child(2)::text").extract()[1].strip()

        # Extract fight statistics
        stats_table = response.css("body > section > div > div > section:nth-child(4) > table > tbody")
        kd_1, kd_2 = stats_table.css("td:nth-child(2) p::text").extract()[0:2]
        sig_str_1, sig_str_2 = stats_table.css("td:nth-child(3) p::text").extract()[0:2]
        total_str_1, total_str_2 = stats_table.css("td:nth-child(5) p::text").extract()[0:2]

        # Store fight data
        fights.append({
            "fight_link": response.url,
            "date": date,
            "location": location,
            "method": method,
            "round": round_num,
            "time": time,
            "time_format": time_format,
            "referee": referee,
            "details": details,
            "name_1": name_1,
            "name_2": name_2,
            "stage_name_1": stage_name_1,
            "stage_name_2": stage_name_2,
            "win_loss_1": win_loss_1,
            "win_loss_2": win_loss_2,
            "kd_1": kd_1,
            "kd_2": kd_2,
            "sig_str_1": sig_str_1,
            "sig_str_2": sig_str_2,
            "total_str_1": total_str_1,
            "total_str_2": total_str_2,
        })

### Run the Scrapy Spider

Initialize and start the crawling process:

In [4]:
process = CrawlerProcess()
process.crawl(UfcSpider)
process.start()

2025-03-19 23:06:13 [scrapy.utils.log] INFO: Scrapy 2.11.2 started (bot: scrapybot)
2025-03-19 23:06:13 [scrapy.utils.log] INFO: Versions: lxml 5.3.0.0, libxml2 2.11.7, cssselect 1.2.0, parsel 1.9.1, w3lib 2.2.1, Twisted 24.10.0, Python 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)], pyOpenSSL 24.2.1 (OpenSSL 3.3.2 3 Sep 2024), cryptography 43.0.3, Platform Windows-10-10.0.22631-SP0
2025-03-19 23:06:13 [scrapy.addons] INFO: Enabled addons:
[]
2025-03-19 23:06:13 [py.warnings] WARNING: c:\Users\ahlaw\OneDrive - UBC\Documents\vscode\Projects\UFC_data_webscraping\.venv\Lib\site-packages\scrapy\utils\request.py:254: ScrapyDeprecationWarning: '2.6' is a deprecated value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting.

It is also the default value. In other words, it is normal to get this warning if you have not defined a value for the 'REQUEST_FINGERPRINTER_IMPLEMENTATION' setting. This is so for backward compatibility reasons, but it will change in

## Data Processing

### Load and Inspect Data

First, let's load the scraped data and examine its structure:

In [5]:
# Load the DataFrames
save_path = "C:\\Users\\ahlaw\\OneDrive - UBC\\Documents\\vscode\\Projects\\UFC_data_webscraping\\Data\\Raw\\"
event_links_df = pd.read_csv(save_path + "event_links.csv")
fights_df = pd.read_csv(save_path + "fight_details.csv")

# Inspect the data
print("Event Links Data:")
print(event_links_df.info())
print("\nFight Details Data:")
print(fights_df.info())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\ahlaw\\OneDrive - UBC\\Documents\\vscode\\Projects\\UFC_data_webscraping\\Data\\Raw\\event_links.csv'

### Data Cleaning

Clean and prepare the data for analysis:

In [ ]:
# 1. Clean column names
fights_df.columns = fights_df.columns.str.strip().str.lower().str.replace(" ", "_")

# 2. Handle missing values
fights_df["referee"] = fights_df["referee"].fillna("Unknown")
numeric_cols = fights_df.select_dtypes(include=["float64", "int64"]).columns
fights_df[numeric_cols] = fights_df[numeric_cols].fillna(0)

# 3. Convert dates to datetime format
fights_df["date"] = pd.to_datetime(fights_df["date"], errors="coerce")

# 4. Convert numeric columns
fights_df[numeric_cols] = fights_df[numeric_cols].apply(pd.to_numeric, errors="coerce")

# 5. Drop duplicates
fights_df = fights_df.drop_duplicates()

### Feature Engineering

Create new features for analysis:

In [ ]:
# Calculate total significant strikes
fights_df["total_sig_strikes_1"] = (
    fights_df["sig_head_1"] + fights_df["sig_body_1"] + fights_df["sig_leg_1"]
)
fights_df["total_sig_strikes_2"] = (
    fights_df["sig_head_2"] + fights_df["sig_body_2"] + fights_df["sig_leg_2"]
)

# Convert control time to seconds
def convert_control_time(time_str):
    if isinstance(time_str, str) and ":" in time_str:
        minutes, seconds = map(int, time_str.split(":"))
        return minutes * 60 + seconds
    return 0

fights_df["ctrl_1_seconds"] = fights_df["ctrl_1"].apply(convert_control_time)
fights_df["ctrl_2_seconds"] = fights_df["ctrl_2"].apply(convert_control_time)

## Save Processed Data

Save the cleaned and processed data:

In [ ]:
# Save processed data
fights_df.to_csv(save_path + "wrangled_fight_details.csv", index=False)
event_links_df.to_csv(save_path + "wrangled_event_links.csv", index=False)
print("Processed data saved successfully in:", save_path)

## Data Overview

The processed dataset includes:

### Event Information
- Event dates and locations
- Event URLs and links

### Fight Details
- Fight methods and outcomes
- Round information and duration
- Referee assignments

### Fighter Information
- Fighter names and nicknames
- Win/loss records
- Fight statistics (knockdowns, strikes)

### Derived Features
- Total significant strikes
- Control time in seconds

### Data Files
1. `event_links.csv`: Raw event URLs
2. `fight_details.csv`: Raw fight statistics
3. `wrangled_event_links.csv`: Processed event data
4. `wrangled_fight_details.csv`: Processed fight statistics